# AI: Delegate the full problem {#sec-delegate-the-full-problem}

![](images/under-construction.png){width=20%}

By now I am sure you appreciate the usefulness of tests. You have been using the tests I wrote for the programming projects, but you have also written a few of your own. 



Your code is wrong if it does not pass all the tests. 



You code is correct if it passes all tests.



Tests made correctly Because tests fail if the code does not passA set of testYou can think of tests as *testing*




# AI: Tests are the contract {#sec-tests-are-the-contract}

You already know how to write a test. In the previous part of the course you learned to make a claim about a function in a form the machine can check, by writing a statement like assert gc_content('GGCC') == 1.0 and letting Python confirm or deny it. This week that skill stops being a way to check your own code and becomes the single most important tool you have for working with an assistant. The idea to hold on to is short: a test is a specification you can execute, and a specification is a contract. Once you see tests that way, the whole business of getting code out of a machine you cannot fully trust becomes manageable, because the test, and not the assistant and not even you, becomes the thing that decides whether the code is right.


Start with the word specification, because it is doing quiet work. Before you can judge whether a piece of code is correct, you have to have decided what correct would mean, and that decision is separate from, and comes before, any code. If you ask an assistant for a function that finds genes, the request has no truth value at all. There is nothing to be right or wrong about, because you have not said what output the function should produce for what input. A specification is exactly that missing thing: a statement of what the result must be for given inputs. When it is written in prose, as a comment or a docstring, it is a promise about what the code will do. When it is written as a test, it is a promise the machine will enforce. That difference is everything. A comment that says the function returns the reverse complement is a promise the code can quietly break, and you saw last week how casually generated code breaks exactly such promises. A test that says assert reverse_complement('ATGC') == 'GCAT' is a promise that cannot be quietly broken, because the moment the code fails to keep it the machine says so.

This gives you a discipline, and the discipline is mostly about order. The natural way to work with an assistant is to ask for the code, receive it, glance at it, and hope. That order puts hope where proof should be. The disciplined order is the reverse: first you decide the specification by choosing concrete examples, then you write those examples as tests, and only then do you ask the assistant for the code, and finally you run the tests to see whether the code it produced actually satisfies the contract you wrote. Writing the tests before you see the assistant's code matters for a reason that is easy to underestimate. If you write the tests afterward, having already seen the plausible looking function the assistant produced, you will, without meaning to, write tests that the function happens to pass. The code will anchor your idea of what correct means, and you will end up checking that the code does what the code does, which proves nothing. Writing the tests first keeps the decision about what correct means where it belongs, with you, made before the assistant had any chance to bias it.

Turning a vague request into a specification is done by choosing examples and working out their answers by hand. Suppose you want the fraction of bases in a strand that are G or C, the GC content. You do not begin by prompting; you begin by writing down that gc_content('GGCC') should be 1.0, that gc_content('ATAT') should be 0.0, and that gc_content('GGAT') should be 0.5, each of which you can compute yourself with certainty. Each such example is one point of the specification, and each becomes one line of your test. The strength of the contract you are writing is exactly the strength of the examples you choose. If every example is gentle, the contract is weak, and a wrong function can satisfy it while still being wrong, which is the happy path trap from last week wearing a different hat. A strong contract includes the awkward inputs on purpose: the empty string, a single base, lowercase letters, a character that should not appear. The nastier your examples, the stronger the promise the machine will hold the code to.

There is a further reason to write the tests first, which is that the act of writing them forces you to make decisions you would otherwise leave to the assistant to guess. The interesting decisions in a specification almost always live in the awkward cases, and the awkward cases are exactly the ones a fluent, confident assistant will silently decide for you, in whatever way its training happened to lean. Consider a function is_gc_rich that is supposed to say whether a strand is GC rich. What should it return for a strand that is exactly half G and C, like GGAT, whose GC content is exactly 0.5? Is a strand that is exactly half GC rich, or not? There is no universal answer; it is a decision, and it is your decision to make. If you write the test first, you are forced to decide it, and you write down assert is_gc_rich('GGAT') == False, or True, deliberately. If you do not, the assistant will pick one silently, using a greater than where you might have wanted a greater than or equal, and you will have no way of knowing whether its choice matches your intention, because you never formed an intention. The same thing happens with a function that returns the most common base in a strand. What should it return when two bases are tied, as G and C are in the strand GC? Run this and watch:

```python
def most_common_base(dna):
    return max(set(dna), key=dna.count)

print(most_common_base('GC'))
```

#### Exercise

[SOLO](../intro/course-introduction.qmd#sec-badge-solo){.small}

Before you run that, write down which base you expect to come back, and the reason you expect it. Then run it. Now the real question, which is not about the answer at all: could you have predicted it from the code alone, without knowing how `set` happens to order its elements and how `max` happens to break a tie? If the answer is no, then the behaviour of this function on a tie is not something you decided; it is something that fell out. Write the single `assert` you would have to add to take that decision back, and notice that writing it forces you to make the decision you had been avoiding.

The answer you get back is essentially arbitrary, an accident of how max happens to break the tie over a set whose order is not something you should rely on. The point is not which base it returns. The point is that the tie is a real decision about what the function should mean, the assistant will make that decision for you and never mention it, and the only way to take the decision back is to write a test that states what you want to happen on a tie before any code exists. Writing tests first is, in this sense, not really about testing at all. It is about doing the thinking, in advance, about the cases where thinking is actually required.

This is why it is worth calling a test a contract rather than merely a check. A contract says: I do not care how you do the job, but the result must satisfy these conditions. That is the relationship you want with an assistant that you cannot fully trust and whose inner workings you cannot inspect. You own the contract, you write it in a form the machine enforces, the assistant supplies an implementation, and the contract, not your hope and not the assistant's confidence, decides whether the implementation is acceptable. A useful consequence of this arrangement is that the implementation becomes replaceable. Because the contract is separate from any particular code, you can throw away the assistant's version and write your own, or accept a faster version later, or fix a bug and try again, and the same contract keeps guarding you the whole time. The tests outlive any one attempt at the code, which is exactly what you want when the code is coming from a source that gets things wrong.

You have already been on the receiving end of this idea without writing the contracts yourself. When you ran the check widget on a project, the tests had been written for you, and every tick and cross you saw was a clause of a contract being enforced against your code. This week you begin writing those contracts yourself, and you meet tests in three ways that will recur for the rest of the course. You will keep using the ready made contracts through the widget when you work on the projects. You will start writing your own assert statements to pin down functions you or the assistant have written. And you will run those tests directly with the pytest command in the terminal and read its report, which lists exactly which contracts held and which were broken, so that you are no longer dependent on a friendly widget to tell you and can read the raw verdict for yourself.

Put the whole discipline together on a small problem before you rely on it. Here is a specification for gc_content written as a contract, before any implementation exists, with the awkward cases decided on purpose:

```python
assert gc_content('GGCC') == 1.0
assert gc_content('ATAT') == 0.0
assert gc_content('GGAT') == 0.5
assert gc_content('ggcc') == 1.0     # lowercase counts too: a deliberate decision
```

Only now do you ask the assistant for the function. If it hands you a version that walks through the strand counting bases equal to the capital letters G and C, that version will satisfy the first three clauses and fail the fourth, and the failure is not a nuisance; it is the contract doing its job, catching a decision the assistant made silently that does not match the decision you made deliberately. Without the fourth clause you would never have known the two of you disagreed.

#### Exercise

[SOLO](../intro/course-introduction.qmd#sec-badge-solo){.small}

Take a vague request and turn it into a contract. Choose a small function you might ask an assistant to write, such as one that counts how many times a given codon appears in a longer strand, and before writing or requesting any code, write down three concrete examples with the answers worked out by hand, then turn each into an `assert` statement. Make sure at least one of your three examples is an awkward one, such as an overlapping occurrence or a codon that does not appear at all, and be ready to say what decision that awkward example forces. Keep this contract; the next three exercises all use it.

#### Exercise

[SOLO](../intro/course-introduction.qmd#sec-badge-solo){.small}

Now with no assistant at all, so that the mechanics are firmly yours. Take a function you have already written earlier in the course and write two tests for it: one that you are confident it passes, and one that you expect it to fail, perhaps because it exposes a case the function never handled. Run both. The point is to feel the difference between a test that confirms and a test that accuses, because a contract that only ever confirms is not protecting you from anything.

#### Exercise

[AI: Drafter](../intro/course-introduction.qmd#sec-badge-drafter){.small}

Run the full discipline in order, on the contract you wrote two exercises ago. You have the specification already, so now, and only now, ask the assistant to write the function. Give it the purpose and the signature. Do **not** give it your tests. Run your tests against what it produces. If they all pass, you have earned a limited and specific belief in the code, limited to the cases you actually specified. If one fails, notice that you found the disagreement before the code ever ran in anger, which is the entire reason for writing the contract first.

#### Exercise

[AI: Drafter](../intro/course-introduction.qmd#sec-badge-drafter){.small}

Now aim the contract at a decision rather than at a bug. Write a contract for `is_gc_rich(dna)`, a function that says whether a strand is GC rich, and make it decide two things the request itself leaves open: what happens at exactly one half, as in `'GGAT'`, and what happens for the empty strand. Decide both deliberately and write an `assert` for each. Then ask the assistant for the function using only the prose description, saying nothing about either case. Run your contract. Whatever it guessed, it guessed silently, and the only reason you can now tell whether the two of you agree is that you decided first. Record which of your two decisions it got wrong, if either.

#### Exercise

[AI: Drafter](../intro/course-introduction.qmd#sec-badge-drafter){.small}

Last, watch the contract outlive the code. Take the same specification and ask the assistant for a *second*, deliberately different implementation of the same function, perhaps by asking for one that avoids a loop, or one that a beginner would find easier to read. Run your unchanged tests against it. Two different pieces of code, one contract, the same verdict. That is what it means to say the implementation is replaceable and the specification is not, and it is why the tests are worth more than any particular function you will ever get out of a machine.

For your logbook this week, record one specification you wrote as tests before requesting any code, note in particular which awkward case you had to make a deliberate decision about, and say whether the assistant's eventual code agreed with your decision or quietly disagreed with it. If it agreed on everything, add the one further awkward case you wish you had thought to specify.



You now hold three separate skills. You can specify a function by writing its contract as tests before any code exists. You can read a function the assistant produced and judge it against a short list of questions, refusing to believe it until the machine agrees. And you can take a task larger than one function, break it into named and testable pieces, and write that plan down before you prompt for anything. The finale is where these three stop being separate exercises and become one continuous way of working, applied to a task that is deliberately larger than anything you could sit down and write on your own. That last point is not an accident or a cruelty; it is the entire purpose of the exercise. The role you are stepping into this week is the one the whole course has been climbing toward, which is to delegate a real job to the assistant and remain completely responsible for the result.

Be exact about what beyond your unaided reach actually means, because it is easy to misread it as a demand that you suddenly become a better programmer than you are. It is the opposite. You are not expected to be able to write the finale from a blank page. You are expected to be able to produce it, correct and understood, by driving the assistant through a plan you designed and by verifying every piece it hands back. The gap between those two things, between writing a program yourself and producing a correct program you did not type, is the gap this course exists to close. A professional programmer would write the finale unaided. You will instead decompose it, specify it, delegate the typing of each piece, and check everything, and if you do that well the finished program will be just as correct as the professional's, and you will understand it just as well, even though you could not have produced it keystroke by keystroke. That is a considerable and useful ability, and it is the one you are being asked to demonstrate.

The way you do it is simply the full loop assembled from the parts you already have, run once for each piece of the plan. You begin, before prompting anything, by decomposing the task into named functions with stated inputs and outputs, exactly as you practiced, so that you are holding a plan made of small pieces each of which you could read, test, and understand on its own. Then you take the pieces in dependency order, and for each one you run a short cycle. You write its contract as one or a few tests, seeded from the example in your plan and stiffened with the awkward cases you have learned to fear. You prompt the assistant for that one function, giving it only what it needs, which is the function's purpose, its inputs and outputs, and the tests it must satisfy. You read what it returns against your judging questions. You run the tests and let the machine deliver the verdict. If the tests pass, that piece is done and you move to the next, building it on top of a piece you have already proven. If the tests fail, you do not start over and you do not despair; you write a specific change request, naming the failing input, the wrong output, the right output, and the reason as far as you can see it, and you send it back and run the tests again. Nothing in that cycle is new. The only new thing is that you are running it many times, on a task whose finished form you could not have written, and watching a program you understand assemble itself out of pieces you have each checked.

The iteration inside a single piece deserves a word, because it is where students most often lose their footing. When a piece fails and you send the assistant a specific change request, the loop should converge: the next version should be closer, and after one or two rounds the tests should pass. If instead the assistant keeps failing in the same way, or fixes your named case while breaking another, that is not a reason to keep hammering the same prompt harder. It is a signal, and it usually means one of two things. Either your specification is wrong or incomplete, so the assistant is optimizing for a target you did not actually mean, in which case you fix the tests and not the code. Or the piece is small enough and you now understand it well enough that you should simply write it yourself, because you have reached the point where delegating costs more than doing. Knowing when to stop asking the assistant and take the keyboard back is part of the skill, not a failure of it. The assistant is there to save you effort on pieces you have specified, not to be argued with indefinitely about a piece you could finish in five minutes.

Two things about verification change scale in the finale and you should meet them deliberately. The first is that the project comes with a suite of tests already written, the same kind you have been running through the check widget, and it is tempting to treat passing that suite as the finish line. It is not. The provided tests are a floor, not a ceiling. They are the conditions the graders and the author knew to check, and passing them is necessary, but it is not proof, because no suite tests every input and the assistant is expert at producing code that satisfies the visible tests while failing the invisible ones. So you add your own tests, especially the nasty cases you have learned to distrust, and you treat the provided suite as the beginning of your checking rather than the end of it. The second is that passing tests, even your own, is still necessary rather than sufficient. A green result tells you the code is correct on the cases you thought of. It cannot tell you the code is correct on the cases you did not. The backstop for everything the tests miss is your own understanding of the pieces, which is why the rule that you never accept code you cannot read matters even more here than it did on a single function. If a function passes every test but you cannot explain why it works, you have not finished with it, because the day it meets an input none of your tests covered, your understanding is the only thing standing between a wrong answer and a wrong result you publish.

That last point is really about responsibility, and responsibility is the part of delegation that does not transfer. You will hand pieces of typing to the assistant, and that is fine, but you cannot hand it the ownership of the result. When you submit the finale, you are submitting it, not the assistant, and you must be able to stand behind every function in it and explain what it does and why it is correct. If a reviewer, or a marker, or your future self points at a function and asks how it works, the answer the AI wrote it is not an answer, it is a confession that you delegated the thinking along with the typing. The whole architecture of the course, the specifying and the judging and the planning, exists so that you can use the assistant heavily and still own the outcome completely. Delegation done well is not the absence of responsibility. It is responsibility exercised through verification instead of through typing.

To make your verification visible, both to yourself and for assessment, you keep a prompt journal alongside the finale as you build it. Where the weekly logbook is a broad reflection, the prompt journal is narrow and practical: a running record, piece by piece, of what you asked the assistant for, where its answer was wrong, how you caught it, and what you did about it. Keeping it costs a little effort in the moment and pays back twice. It forces you to actually notice the failures rather than skimming past a bug you half saw, and by the end of the project it is a concrete record of your own judgment at work, which is exactly the thing the finale is meant to demonstrate. A finale that runs is worth less, in this course, than a finale that runs and comes with a frank account of the half dozen places the assistant got it wrong and how you knew.

You will do most of this on the real project, so the exercises here are small and meant to sharpen particular moves in the loop.

#### Exercise

[AI: Developer](../intro/course-introduction.qmd#sec-badge-developer){.small}

Run one full cycle on one piece, in the order that makes it safe. Before you prompt for anything, write the acceptance criteria for a single function from your plan: what it must return for which inputs, including the awkward cases, in a form the machine can check. Then prompt the assistant for that function, giving it the purpose, the signature, and nothing you have not decided. Read what comes back. Run the criteria. The thing to notice is that you wrote the criteria while you still had no code in front of you to anchor your idea of what correct meant, which is the only condition under which delegating a whole piece is anything other than hoping.

#### Exercise

[AI: Developer](../intro/course-introduction.qmd#sec-badge-developer){.small}

Take a single function from your plan, prompt the assistant for it, and when its first attempt fails one of your tests, write the change request you would send back, containing the failing input, the wrong output, the correct output, and the reason. Notice how much of that you already knew from having written the test first. Send it. If the second version passes, the loop converged; if the third does not, stop, and go to the next exercise.

#### Exercise

[SOLO](../intro/course-introduction.qmd#sec-badge-solo){.small}

Take the keyboard back. Find a piece where the loop did not converge in two rounds, or where you can already see the answer, and write it yourself. Time both if you can. The skill this exercise trains is not programming, it is the judgment about when delegating has stopped paying, and it is a skill in exactly the same sense that specifying and judging are. An assistant argued with for twenty minutes over a function you could have typed in five is not saving you anything, and noticing that quickly is part of using it well.

#### Exercise

[SOLO](../intro/course-introduction.qmd#sec-badge-solo){.small}

Take a function that has just passed the provided tests and write one more test of your own, a nasty case the provided suite did not include, and run it. If it also passes, you have strengthened the contract. If it fails, you have just caught a bug that would otherwise have shipped, and you have proved to yourself that the provided suite was a floor and not a ceiling.

#### Exercise

[AI: Developer](../intro/course-introduction.qmd#sec-badge-developer){.small}

Finally, the exercise that decides whether you may keep the badge. Pick the function in your finale that you delegated most heavily, the one you are least sure you could have written yourself, and write one paragraph explaining how it works, in your own words, with the assistant closed. Not what it does, which you know from its tests, but *how* it does it, line by line. If you cannot, that function is not finished, whatever its tests say, because you cannot own a result you cannot explain. Do the reading now and write the paragraph again. This is what it means to say that delegation transfers the typing and not the responsibility.

#### Exercise

[SOLO](../intro/course-introduction.qmd#sec-badge-solo){.small}

Write one prompt journal entry for a piece you built, recording where the assistant was wrong and how the machine, or your reading, revealed it.

For your logbook this week, step back from the individual pieces and describe the experience of producing something you could not have written unaided: which part of the loop, the planning, the specifying, the judging, or the iterating, did the most work in making the task possible, and name one function in your finale that you are confident you could now explain in full to someone else, and one that you are not yet sure you could, because the second kind is where your remaining work is.